In [16]:
"""
Inside vs outside similarity per group: table + bar chart + simple coin map.

INPUT : similarity_outputs/composite_similarity_matrix.csv
        similarity_outputs/cluster_assignments_K4.csv
OUTPUT: similarity_outputs/inside_outside_table.csv
        similarity_outputs/fig_36_inside_outside_bars.png
        similarity_outputs/fig_36_group_map.png
"""
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.manifold import MDS

OUTPUT_DIR = "similarity_outputs"
SIM_PATH = f"{OUTPUT_DIR}/composite_similarity_matrix.csv"
ASSIGN_PATH = f"{OUTPUT_DIR}/cluster_assignments_K4.csv"
N_MAP_COINS = 5                                   # example coins per group on the map
FORCE = ["bitcoin", "ethereum"]                   # always shown if present
COL = {1: "#2a7de1", 2: "#d64545", 3: "#2f9e5b", 4: "#e08a00"}


def inside_outside(sim, labels):
    """Average similarity of pairs inside the group vs pairs between the group and all other coins."""
    S = sim.values
    rows = []
    for k in sorted(np.unique(labels)):
        m = np.where(labels == k)[0]
        o = np.where(labels != k)[0]
        inside = S[np.ix_(m, m)][np.triu_indices(len(m), 1)].mean()
        outside = S[np.ix_(m, o)].mean()
        rows.append({"group": int(k), "coins": len(m),
                     "inside": round(inside, 2), "outside": round(outside, 2)})
    return pd.DataFrame(rows)


def bar_chart(tab, path):
    x = np.arange(len(tab))
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    ax.bar(x - 0.2, tab["inside"], 0.4, label="Inside the group", color="#2a7de1")
    ax.bar(x + 0.2, tab["outside"], 0.4, label="Outside (other groups)", color="#b8b8b8")
    for i, r in tab.iterrows():
        ax.text(i - 0.2, r["inside"] + 0.01, f"{r['inside']:.2f}", ha="center", fontsize=10)
        ax.text(i + 0.2, r["outside"] + 0.01, f"{r['outside']:.2f}", ha="center", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Group {g}\n({n} coins)" for g, n in zip(tab["group"], tab["coins"])])
    ax.set_ylabel("Average similarity (0 to 1)")
    ax.set_ylim(0, max(tab["inside"].max(), tab["outside"].max()) + 0.12)
    ax.set_title("Inside vs outside similarity", loc="left", fontsize=13)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(path, facecolor="white")
    plt.close(fig)


def group_map(sim, assign, path):
    """Example coins only: most typical coins of each group (+ forced names). Closer dots = more alike."""
    chosen = []
    for k in sorted(assign.unique()):
        m = assign.index[assign == k]
        sub = sim.loc[m, m].values.copy()
        np.fill_diagonal(sub, np.nan)
        core = pd.Series(np.nanmean(sub, axis=1), index=m).sort_values(ascending=False)
        pick = [c for c in FORCE if c in m][:N_MAP_COINS]
        pick += [c for c in core.index if c not in pick][:N_MAP_COINS - len(pick)]
        chosen += pick
    D = 1 - sim.loc[chosen, chosen].values
    np.fill_diagonal(D, 0)
    xy = MDS(n_components=2, dissimilarity="precomputed", random_state=3,
             n_init=8, max_iter=1000).fit_transform(D)
    df = pd.DataFrame(xy, index=chosen, columns=["x", "y"])
    df["g"] = assign.loc[chosen].values

    fig, ax = plt.subplots(figsize=(10, 6.5), dpi=150)
    for k, s in df.groupby("g"):
        ax.scatter(s.x, s.y, s=110, color=COL.get(int(k), "#666"), edgecolor="white",
                   linewidth=1.3, label=f"Group {int(k)}", zorder=3)
        for c, r in s.iterrows():
            ax.annotate(c, (r.x, r.y), xytext=(6, 5), textcoords="offset points", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("Map direction X (no units)"); ax.set_ylabel("Map direction Y (no units)")
    ax.set_title("Each dot is a coin. Closer dots behave more alike.", loc="left", fontsize=13)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def run(sim_path=SIM_PATH, assign_path=ASSIGN_PATH, outdir=OUTPUT_DIR):
    sim = pd.read_csv(sim_path, index_col=0)
    assign = pd.read_csv(assign_path).set_index("coin_id")["cluster"].reindex(sim.index)
    assert assign.notna().all(), "some coins in the matrix have no group"
    tab = inside_outside(sim, assign.values)
    tab.to_csv(f"{outdir}/inside_outside_table.csv", index=False)
    print(tab.to_string(index=False))
    bar_chart(tab, f"{outdir}/fig_36_inside_outside_bars.png")
    group_map(sim, assign, f"{outdir}/fig_36_group_map.png")
    return tab


if __name__ == "__main__":
    run()

 group  coins  inside  outside
     1    219    0.28     0.22
     2      9    0.08     0.16
     3     38    0.24     0.24
     4     24    0.12     0.18


In [12]:
"""
Single methods vs composite: cluster each similarity matrix the same way (Ward, K = 4)
and score all of them with measures that none of the methods was built for.

INPUT : similarity_outputs/*.csv  (the 5 single matrices + composite, from Section 3.4)
        returns: ret_pivot if it exists in the notebook, otherwise loaded from DATA_PATH below
OUTPUT: similarity_outputs/single_vs_composite_table.csv
        similarity_outputs/fig_37_single_vs_composite.png   (3 small bar charts)
        similarity_outputs/fig_37_grouping_agreement.png    (ARI heatmap)
"""
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score

OUTPUT_DIR = "similarity_outputs"
DATA_PATH = "../data/crypto_with_indicators.parquet"   # same as in 3_4
MAX_COINS = 1000
MIN_DAYS = 180
K = 4
BTC_ID = "bitcoin"
FILES = {                                   # name -> csv file
    "Copula":        "copula_ltdc_matrix.csv",
    "HMM":           "hmm_similarity_matrix.csv",
    "GLASSO":        "glasso_similarity_matrix.csv",
    "DTW":           "dtw_similarity_matrix.csv",
    "Cointegration": "coint_similarity_matrix.csv",
    "Composite":     "composite_similarity_matrix.csv",
}


def ward_groups(sim, k=K):
    D = 1 - sim.values.astype(float)
    D = (D + D.T) / 2
    np.fill_diagonal(D, 0)
    D = np.clip(D, 0, None)
    Z = linkage(squareform(D, checks=False), method="ward")
    return fcluster(Z, k, criterion="maxclust")


def eta_squared(values, labels):
    """Share of the differences in a coin attribute that is explained by the group (0 to 1)."""
    v = pd.Series(values).astype(float)
    ok = v.notna().values
    v, g = v[ok].values, np.asarray(labels)[ok]
    sst = ((v - v.mean()) ** 2).sum()
    ssb = sum(len(v[g == k]) * (v[g == k].mean() - v.mean()) ** 2 for k in np.unique(g))
    return ssb / sst if sst > 0 else np.nan


def score(labels, coins, corr, vol, corr_btc):
    iu = np.triu_indices(len(coins), 1)
    c = corr.loc[coins, coins].values[iu]
    same = labels[iu[0]] == labels[iu[1]]
    return {
        "group_sizes": "/".join(str(n) for n in sorted(np.bincount(labels)[1:], reverse=True)),
        "pearson_inside": np.nanmean(c[same]),
        "pearson_outside": np.nanmean(c[~same]),
        "pearson_gap": np.nanmean(c[same]) - np.nanmean(c[~same]),
        "vol_separation": eta_squared(vol.loc[coins].values, labels),
        "btc_corr_separation": eta_squared(corr_btc.loc[coins].values, labels),
    }


def run(ret, files=FILES, outdir=OUTPUT_DIR, k=K, read=pd.read_csv):
    mats = {n: read(f"{outdir}/{f}", index_col=0) for n, f in files.items()}
    coins = [c for c in mats["Composite"].index if c in ret.columns]
    ret = ret[coins]
    corr = ret.corr()
    vol = ret.std()
    corr_btc = ret.corrwith(ret[BTC_ID]) if BTC_ID in ret.columns else pd.Series(np.nan, index=coins)

    labs, rows = {}, []
    for n, m in mats.items():
        m = m.reindex(index=coins, columns=coins).fillna(0)
        labs[n] = ward_groups(m, k)
        rows.append({"method": n, **score(labs[n], coins, corr, vol, corr_btc)})
    tab = pd.DataFrame(rows).set_index("method")
    tab.round(3).to_csv(f"{outdir}/single_vs_composite_table.csv")
    print(tab.round(3).to_string())

    # --- plot 1: three small bar charts, composite highlighted
    names = list(tab.index)
    colors = ["#2a7de1" if n == "Composite" else "#b8b8b8" for n in names]
    panels = [("pearson_gap", "Inside minus outside\nPearson correlation"),
              ("vol_separation", "How much volatility differs\nbetween groups (0 to 1)"),
              ("btc_corr_separation", "How much link to Bitcoin differs\nbetween groups (0 to 1)")]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), dpi=150, sharey=False)
    for ax, (col, title) in zip(axes, panels):
        vals = tab[col].values
        ax.bar(names, vals, color=colors)
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
        ax.set_title(title, fontsize=10, loc="left")
        ax.tick_params(axis="x", rotation=40, labelsize=8)
        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)
    fig.suptitle(f"Single methods vs composite (same Ward clustering, K = {k}). Higher = groups differ more", x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{outdir}/fig_37_single_vs_composite.png", facecolor="white")
    plt.close(fig)

    # --- plot 2: do the groupings agree with each other? (ARI, 1 = same groups)
    ari = pd.DataFrame([[adjusted_rand_score(labs[a], labs[b]) for b in names] for a in names],
                       index=names, columns=names)
    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
    im = ax.imshow(ari.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=40, ha="right", fontsize=9)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=9)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{ari.values[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if ari.values[i, j] > 0.55 else "black")
    ax.set_title("Do the groupings agree? (1 = identical groups, 0 = no better than chance)", fontsize=10, loc="left")
    plt.colorbar(im, fraction=0.046)
    plt.tight_layout()
    plt.savefig(f"{outdir}/fig_37_grouping_agreement.png", facecolor="white")
    plt.close(fig)
    return tab, ari


def load_returns(path=DATA_PATH, max_coins=MAX_COINS, min_days=MIN_DAYS):
    """Same steps as load_and_prepare in 3_4: no stablecoins, top N by latest market cap,
    log returns, drop coins with fewer than min_days observations."""
    if path.endswith(".parquet"):
        try:
            import polars as pl
            df = pl.scan_parquet(path).collect().to_pandas()
        except ImportError:
            df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, parse_dates=["date"])
    df["date"] = pd.to_datetime(df["date"])
    df = df[df["is_stablecoin"] == False].copy()
    if "log_ret" not in df.columns:
        df = df.sort_values(["coin_id", "date"])
        df["log_ret"] = df.groupby("coin_id")["close"].transform(lambda x: np.log(x / x.shift(1)))
    top = (df.groupby("coin_id").agg(market_cap=("market_cap_usd", "last"))
             .sort_values("market_cap", ascending=False).head(max_coins).index)
    df = df[df["coin_id"].isin(top)]
    ret = df.pivot_table(index="date", columns="coin_id", values="log_ret")
    ret = ret.loc[:, ret.count() >= min_days]
    print(f"Returns loaded: {ret.shape[1]} coins, {ret.shape[0]} days")
    return ret


if __name__ == "__main__":
    # Works in any notebook or script: reuses ret_pivot if it exists in memory,
    # otherwise loads the returns itself from DATA_PATH.
    run(ret_pivot if "ret_pivot" in globals() else load_returns())

Returns loaded: 615 coins, 1823 days
                group_sizes  pearson_inside  pearson_outside  pearson_gap  vol_separation  btc_corr_separation
method                                                                                                        
Copula         128/90/62/10           0.503            0.424        0.078           0.036                0.408
HMM             166/112/8/4           0.461            0.442        0.019           0.375                0.200
GLASSO          204/65/12/9           0.422            0.485       -0.064           0.022                0.297
DTW            154/68/58/10           0.506            0.417        0.089           0.085                0.452
Cointegration  103/96/56/35           0.454            0.450        0.004           0.033                0.044
Composite      168/80/22/20           0.469            0.437        0.032           0.030                0.135
